# Foreign Whispers — Duration-Aware Dubbing Pipeline

This notebook covers the full project arc from basic pipeline to duration-aware dubbing:

| Milestone | Description |
|-----------|-------------|
| M0 | Baseline analysis — understand current timing failures |
| M1 | Download video + captions |
| M1-align | Speech activity detection (VAD) |
| M2-align | Speaker diarization |
| M2 | Speech-to-text via Whisper |
| M3 | Translate EN → ES (argostranslate, offline) |
| M3-align | Instrument segment timing metrics |
| M4-align | Explicit alignment fallback policy |
| M5-align | Duration-aware translation re-ranking |
| M6-align | Global timeline alignment optimizer |
| M4 | Text-to-speech with soft duration targets |
| M5 | Stitch audio + subtitles into output video |
| M8-align | Evaluation and experiment comparison |

**PydanticAI** agents are offered as options in M3, M5-align, M6-align, and M8-align.
**Logfire** spans are offered as options at every stage.

```
YouTube URL
  → [M1]        Download video + captions
  → [M1-align]  Speech Activity Detection (VAD)
  → [M2-align]  Speaker Diarization
  → [M2]        Whisper Transcription
  → [M3]        Translation EN → ES
  → [M3-align]  Segment Metrics Instrumentation
  → [M4-align]  Fallback Policy
  → [M5-align]  Duration-Aware Re-ranking  (PydanticAI)
  → [M6-align]  Global Alignment Optimizer (PydanticAI)
  → [M4]        TTS with soft duration targets
  → [M5]        Stitch audio + subtitles
```

---
## Requirements

```bash
uv sync
# System: sudo apt install ffmpeg imagemagick
# Optional: pip install pydantic-ai logfire pyannote.audio silero-vad nest_asyncio
```

## Setup — paths and imports

In [ ]:
import sys
import json
import pathlib

REPO_ROOT = pathlib.Path("..")
sys.path.insert(0, str(REPO_ROOT))

DATA_DIR         = REPO_ROOT / "pipeline_data"
RAW_VIDEO_DIR    = DATA_DIR / "raw_videos"
RAW_CAPTION_DIR  = DATA_DIR / "raw_captions"
TRANSCRIPTION_EN = DATA_DIR / "transcriptions" / "en"
TRANSCRIPTION_ES = DATA_DIR / "transcriptions" / "es"
AUDIO_DIR        = DATA_DIR / "audios"
OUTPUT_VIDEO_DIR = DATA_DIR / "output_videos"
METRICS_DIR      = DATA_DIR / "metrics"

for d in [RAW_VIDEO_DIR, RAW_CAPTION_DIR, TRANSCRIPTION_EN,
          TRANSCRIPTION_ES, AUDIO_DIR, OUTPUT_VIDEO_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Directories ready.")

### Option: Logfire — configure tracing

Skip this cell if not using Logfire.  When enabled, every pipeline span is recorded
with structured fields for segment-level debugging and experiment comparison.

```bash
pip install logfire
logfire auth    # authenticate once
```

In [ ]:
# Optional — remove the try/except guard once logfire is installed
try:
    import logfire
    logfire.configure(service_name="foreign-whispers")
    LOGFIRE_ENABLED = True
    print("Logfire tracing enabled.")
except ImportError:
    import contextlib

    class _NoopSpan:
        def __enter__(self): return self
        def __exit__(self, *a): pass

    class _noop:
        @staticmethod
        def span(name, **kw): return _NoopSpan()
        @staticmethod
        def info(*a, **kw): pass

    logfire = _noop()
    LOGFIRE_ENABLED = False
    print("Logfire not installed — using no-op shim.")

---
## M0 — Baseline Analysis

Understand precisely where timing enters, where it is ignored, and where it is enforced
before changing anything.

| Question | Answer |
|----------|--------|
| Where does timing enter? | `transcribe.py` → Whisper `segments[].start/end` |
| Where is timing ignored? | `translate_en_to_es.py` — only `text` rewritten, timestamps unchanged |
| Where is timing enforced? | `tts_es._synced_segment_audio()` — post-hoc stretch via pyrubberband |
| What knows nothing about duration? | `XTTSClient.tts_to_file()`, `translate_sentence()` |

In [ ]:
# M0: trace the key contracts to understand where timing enters and is lost
import inspect
from tts_es import _synced_segment_audio
from translate_en_to_es import translate_file

print("=== _synced_segment_audio signature ===")
print(inspect.signature(_synced_segment_audio))
print()
print("=== translate_file signature ===")
print(inspect.signature(translate_file))
print()
print("Observation: translate_file has no duration_budget parameter.")
print("Observation: _synced_segment_audio receives target_sec — but only AFTER synthesis.")

---
## M1 — Download Video and Captions

Reads URLs from `video_registry.yml` — the single source of truth for all registered videos.

In [ ]:
import yaml
from download_video import download_video, download_caption, get_video_info

# Load the single source of truth for registered videos
with open(REPO_ROOT / "video_registry.yml") as f:
    registry = yaml.safe_load(f)

VIDEOS = registry["videos"]
print(f"Registered videos: {len(VIDEOS)}")
for v in VIDEOS:
    print(f"  {v['id']}  {v['title']}")

# Pick one to demo the single-video cells below
VIDEO_URL = VIDEOS[0]["url"]
video_id, title = get_video_info(VIDEO_URL)
print(f"\nSelected  → {video_id}: {title}")

In [ ]:
with logfire.span("download", video_id=video_id):
    video_path   = download_video(VIDEO_URL, str(RAW_VIDEO_DIR))
    caption_path = download_caption(VIDEO_URL, str(RAW_CAPTION_DIR))

print(f"Video   : {video_path}")
print(f"Captions: {caption_path}")

### Inspect captions

Each line is a JSON object: `{"text": str, "start": float, "duration": float}`

In [ ]:
import json

with open(caption_path) as f:
    caption_segments = [json.loads(line) for line in f if line.strip()]

print(f"Caption segments: {len(caption_segments)}")
for seg in caption_segments[:5]:
    print(seg)

---
## M1-align — Speech Activity Detection (VAD)

Replaces the implicit notion of silence with an explicit speech / silence timeline.
This makes silence budgets visible to the global alignment optimizer (M6-align) instead
of being inferred from Whisper segment gaps.

**Library:** [Silero VAD](https://github.com/snakers4/silero-vad) (MIT, offline, fast)

### Option: Logfire span — records speech region count and total silence budget

In [ ]:
import subprocess, tempfile
from foreign_whispers import detect_speech_activity

# Extract mono 16 kHz audio for VAD
audio_tmp = pathlib.Path(tempfile.mktemp(suffix=".wav"))
subprocess.run(
    ["ffmpeg", "-y", "-i", video_path, "-ac", "1", "-ar", "16000", "-vn", str(audio_tmp)],
    check=True, capture_output=True,
)

with logfire.span("speech_activity", video_id=video_id):
    speech_timeline = detect_speech_activity(str(audio_tmp))

speech_regions  = [r for r in speech_timeline if r["label"] == "speech"]
silence_regions = [r for r in speech_timeline if r["label"] == "silence"]
total_silence   = sum(r["end_s"] - r["start_s"] for r in silence_regions)
print(f"Speech regions : {len(speech_regions)}")
print(f"Silence regions: {len(silence_regions)}  ({total_silence:.1f}s available for overflow)")

---
## M2-align — Speaker Diarization

Attach speaker-turn labels to the timeline so alignment can avoid splitting dubbed
speech across a turn boundary.

**Library:** [pyannote.audio](https://github.com/pyannote/pyannote-audio) — requires
accepting the model licence on HuggingFace and providing an `HF_TOKEN`.

### Option: Logfire span — records `speaker_id` per segment

In [ ]:
from foreign_whispers import diarize_audio

HF_TOKEN = None  # set to your token: "hf_..."

with logfire.span("speaker_diarization", video_id=video_id):
    speaker_timeline = diarize_audio(str(audio_tmp), hf_token=HF_TOKEN)

speakers = {t["speaker"] for t in speaker_timeline}
print(f"Speaker turns: {len(speaker_timeline)}, unique speakers: {len(speakers)}")
for t in speaker_timeline[:5]:
    print(f"  [{t['start_s']:.1f}s – {t['end_s']:.1f}s]  {t['speaker']}")

---
## M2 — Speech-to-Text via Whisper

Whisper creates the segment timestamps that flow through the entire pipeline.
These are the first source of timing and become ground truth for all alignment decisions.

> Note: timestamps are fixed after this step — translation and TTS currently have no
> visibility into the duration budget until M3-align instruments them.

In [ ]:
from transcribe import load_model, transcribe_videos

# Download/load model once — cached by Whisper after first use
MODEL_SIZE = "tiny"   # options: tiny | base | small | medium | large
model = load_model(MODEL_SIZE)

In [ ]:
with logfire.span("transcribe", video_id=video_id, model=MODEL_SIZE):
    transcribe_videos(str(RAW_VIDEO_DIR), model, str(TRANSCRIPTION_EN))

en_files = list(TRANSCRIPTION_EN.glob("*.json"))
print(f"Transcripts produced: {len(en_files)}")

### Inspect a transcript

In [ ]:
with open(en_files[0]) as f:
    transcript = json.load(f)

print("Language:", transcript.get("language"))
print(f"Segments: {len(transcript['segments'])}")
print("\nFirst 3 segments:")
for seg in transcript["segments"][:3]:
    dur = seg["end"] - seg["start"]
    print(f"  [{seg['start']:.1f}s – {seg['end']:.1f}s ({dur:.1f}s)]  {seg['text'].strip()}")

---
## M3 — Translate English → Spanish

Baseline translation using **argostranslate** (OpenNMT, offline — no commercial APIs).
Timestamps are preserved; only `text` is rewritten.

> **Key limitation:** the translator has no duration budget parameter.
> `translate_file()` rewrites text but never scores whether the result fits the
> source segment window.  Duration-aware re-ranking is added in M5-align.

In [ ]:
from translate_en_to_es import download_and_install_package, translate_all_files

FROM_LANG = "en"
TO_LANG   = "es"

# Download and install the en→es language pack (one-time, ~100 MB)
download_and_install_package(FROM_LANG, TO_LANG)
print("Language pack ready.")

In [ ]:
with logfire.span("translate", from_lang=FROM_LANG, to_lang=TO_LANG):
    translate_all_files(
        source_directory=str(TRANSCRIPTION_EN),
        destination_directory=str(TRANSCRIPTION_ES),
        from_code=FROM_LANG, to_code=TO_LANG,
    )

es_files = list(TRANSCRIPTION_ES.glob("*.json"))
print(f"Translated files: {len(es_files)}")

### Compare EN vs ES segment

In [ ]:
with open(en_files[0]) as f:
    en_data = json.load(f)
with open(es_files[0]) as f:
    es_data = json.load(f)

print("EN:", en_data["segments"][0]["text"].strip())
print("ES:", es_data["segments"][0]["text"].strip())

---
## M3-align — Instrument Segment Timing Metrics

Make timing mismatch measurable before trying to fix it.  For each segment record:
source duration, translated character count, predicted TTS duration (heuristic),
predicted stretch factor, and overflow.

The `SegmentMetrics` dataclass uses a syllable-based heuristic (~4.5 syllables/second
for Spanish) to predict TTS duration. The `DurationAwareTTSBackend` ABC defines the
extended TTS interface with soft duration targets.

All types and functions are imported from the `foreign_whispers` library.

### Option: Logfire — one `score.segment` span per segment with all fields

In [ ]:
from foreign_whispers import DurationAwareTTSBackend

print("DurationAwareTTSBackend contract imported from foreign_whispers.backends.")
print("Implement in: api/src/inference/tts_local.py  api/src/inference/tts_remote.py")

In [ ]:
from foreign_whispers import (
    AlignAction,
    AlignedSegment,
    SegmentMetrics,
    compute_segment_metrics,
    decide_action,
)

# Load EN and ES transcripts and compute segment metrics
with open(en_files[0]) as f:
    en_transcript = json.load(f)
with open(es_files[0]) as f:
    es_transcript = json.load(f)

all_metrics = compute_segment_metrics(en_transcript, es_transcript)
bad = [m for m in all_metrics if m.predicted_stretch > 1.5]
print(f"Total segments : {len(all_metrics)}")
print(f"Stretch > 1.5x : {len(bad)}  ({100*len(bad)/max(len(all_metrics),1):.0f}%)")
print("\nWorst 5:")
for m in sorted(bad, key=lambda x: -x.predicted_stretch)[:5]:
    print(f"  seg {m.index:3d}  stretch={m.predicted_stretch:.2f}x  overflow={m.overflow_s:.1f}s")
    print(f"    EN: {m.source_text[:55]}")
    print(f"    ES: {m.translated_text[:55]}")

---
## M4-align — Explicit Alignment Fallback Policy

Replace the implicit "always stretch to fit" with a clear decision per segment:

| Stretch factor | Action |
|---------------|--------|
| ≤ 1.1 | **accept** — no stretch needed |
| 1.1 – 1.4 | **mild_stretch** — pyrubberband within perceptually safe range |
| 1.4 – 1.8 | **gap_shift** — borrow from adjacent silence if available |
| 1.8 – 2.5 | **request_shorter** — escalate to translation re-ranking (M5-align) |
| > 2.5 | **fail** — log diagnostic, use silence fallback |

### Option: Logfire — records `policy_action` per segment

In [ ]:
action_counts = {a: 0 for a in AlignAction}
for m in all_metrics:
    action = decide_action(m)
    action_counts[action] += 1
    with logfire.span(
        "align.segment",
        segment_index = m.index,
        stretch       = round(m.predicted_stretch, 3),
        policy_action = action.value,
    ):
        pass

print("Policy distribution:")
for action, count in action_counts.items():
    bar = "█" * count
    print(f"  {action.value:<20} {count:3d}  {bar}")

---
## M5-align — Duration-Aware Translation Re-ranking

For segments where the fallback policy decided `REQUEST_SHORTER`, generate alternative
translations and re-rank them against the duration budget.

### Option: PydanticAI Translation Candidate Agent

Receives source text, baseline translation, target duration, and neighboring context.
Returns typed candidates with brevity rationale.  The deterministic duration scorer
picks the winner — the agent never touches timestamps directly.

```bash
pip install pydantic-ai
export ANTHROPIC_API_KEY=...
```

In [ ]:
from foreign_whispers import get_shorter_translations, TranslationCandidate

try:
    from foreign_whispers.agents import PYDANTICAI_AVAILABLE as PYDANTICAI_ENABLED
    if PYDANTICAI_ENABLED:
        print("PydanticAI translation agent ready (from foreign_whispers.agents).")
    else:
        print("pydantic-ai not installed — agent re-ranking disabled.")
except ImportError:
    PYDANTICAI_ENABLED = False
    print("pydantic-ai not installed — agent re-ranking disabled.")

In [ ]:
import asyncio

# Run re-ranking only for REQUEST_SHORTER segments
with open(es_files[0]) as f:
    es_data = json.load(f)

need_rerank = [m for m in all_metrics if decide_action(m) == AlignAction.REQUEST_SHORTER]
print(f"Segments queued for re-ranking: {len(need_rerank)}")

if PYDANTICAI_ENABLED and need_rerank:
    try:
        import nest_asyncio; nest_asyncio.apply()
    except ImportError:
        pass

    async def run_reranking():
        for m in need_rerank:
            prev = es_data["segments"][m.index - 1]["text"] if m.index > 0 else ""
            nxt  = es_data["segments"][m.index + 1]["text"] if m.index < len(es_data["segments"]) - 1 else ""
            candidates = await get_shorter_translations(
                source_text       = m.source_text,
                baseline_es       = m.translated_text,
                target_duration_s = m.source_duration_s,
                context_prev      = prev,
                context_next      = nxt,
            )
            if candidates:
                best = min(candidates, key=lambda c: abs(len(c.text) / 15.0 - m.source_duration_s))
                es_data["segments"][m.index]["text"] = best.text
                print(f"  seg {m.index}: -> {best.text[:50]!r}")

    asyncio.run(run_reranking())
    reranked_path = es_files[0].with_suffix(".reranked.json")
    with open(reranked_path, "w") as f:
        json.dump(es_data, f)
    print(f"Saved: {reranked_path}")
else:
    print("Skipping re-ranking (agent not enabled or no candidates).")

---
## M6-align — Global Timeline Alignment Optimizer

Stop treating each segment as an isolated timing problem.  Build a timeline model
with segments, silence gaps, and cumulative drift, then apply a global pass that can
shift neighboring segments into available silence rather than forcing local stretches.

### Option: PydanticAI Alignment Policy Agent

The agent chooses among the `AlignAction` options using the measured costs as inputs.
It does not compute the schedule itself — that stays in deterministic Python code.

### Option: Logfire — records `gap_shift_ms` and `cumulative_drift_ms` per segment

In [ ]:
from foreign_whispers import global_align

aligned_segments = global_align(all_metrics, silence_regions)

shifts    = [s for s in aligned_segments if s.action == AlignAction.GAP_SHIFT]
stretches = [s for s in aligned_segments if s.action == AlignAction.MILD_STRETCH]
drift     = aligned_segments[-1].scheduled_end - aligned_segments[-1].original_end if aligned_segments else 0.0

print(f"Total segments : {len(aligned_segments)}")
print(f"Gap shifts     : {len(shifts)}")
print(f"Mild stretches : {len(stretches)}")
print(f"Total drift    : {drift:.2f}s")

---
## M8-align — Evaluation and Experiment Comparison

Report the metrics defined in the alignment design spec for one clip, then optionally
invoke the **PydanticAI Failure Analysis Agent** to cluster failure modes.

### Metrics reported

`mean_abs_duration_error_s`, `pct_severe_stretch`, `n_gap_shifts`,
`n_translation_retries`, `total_cumulative_drift_s`

### Option: PydanticAI Failure Analysis Agent — `claude-opus-4-6`

In [ ]:
from foreign_whispers import clip_evaluation_report, analyze_failures

report = clip_evaluation_report(all_metrics, aligned_segments)
print("=== Clip Evaluation Report ===")
for k, v in report.items():
    print(f"  {k:<40} {v}")

report_path = METRICS_DIR / f"{pathlib.Path(en_files[0]).stem}_eval.json"
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)
print(f"\nSaved: {report_path}")

# Option: PydanticAI Failure Analysis Agent
if PYDANTICAI_ENABLED:
    try:
        import nest_asyncio; nest_asyncio.apply()
    except ImportError:
        pass

    analysis = asyncio.run(analyze_failures(report))
    if analysis:
        print("\n=== Failure Analysis Agent ===")
        print(f"Category    : {analysis.failure_category}")
        print(f"Root cause  : {analysis.likely_root_cause}")
        print(f"Next change : {analysis.suggested_change}")
    else:
        print("\nFailure analysis agent returned no results.")
else:
    print("\npydantic-ai not installed — skipping failure analysis agent.")

---
## M4 — Text-to-Speech (Spanish)

Synthesizes dubbed audio using **Coqui TTS** (local CPU/GPU) or an **XTTS API server**.

Each segment is synthesized then **time-stretched** via `pyrubberband` to fit the source
segment window.  Gaps are filled with silence.  A YouTube-caption timing offset is
applied so dubbed speech starts when the speaker's mouth opens.

### Alignment M7 — extended TTS interface with soft duration targets

The current `tts_to_file(text, output_path)` has no duration parameter.
The cell below shows the extended contract defined in `api/src/inference/base.py` that
lets backends carry alignment intent before waveform generation starts.

### Option: Logfire span — records `raw_tts_duration_ms` and `stretch_factor`

In [ ]:
from tts_es import text_file_to_speech, files_from_dir
import os

# os.environ["XTTS_API_URL"] = "http://localhost:8020"  # prefer GPU server

for es_file in files_from_dir(str(TRANSCRIPTION_ES)):
    with logfire.span("tts", source=pathlib.Path(es_file).name):
        text_file_to_speech(es_file, str(AUDIO_DIR))

wav_files = list(AUDIO_DIR.glob("*.wav"))
print(f"WAV files produced: {len(wav_files)}")

### Listen to a sample

In [ ]:
from IPython.display import Audio

if wav_files:
    Audio(str(wav_files[0]))

---
## Milestone 5 — Stitch audio and subtitles into output video

Combines the original video, the dubbed WAV, and burnt-in Spanish subtitles
using **moviepy** + **ffmpeg**.  TextClip rendering requires ImageMagick.

Two stitching modes are available:
- `stitch_audio` — fast ffmpeg remux (no subtitle burn-in)
- `stitch_video_with_timestamps` — full CompositeVideoClip with subtitles (slower)

In [ ]:
from translated_output import stitch_audio, stitch_video_with_timestamps

video_files = list(RAW_VIDEO_DIR.glob("*.mp4"))
audio_files = list(AUDIO_DIR.glob("*.wav"))

if not video_files or not audio_files:
    print("Run milestones 1-4 first.")
else:
    stem         = video_files[0].stem
    output_path  = str(OUTPUT_VIDEO_DIR / f"{stem}_es.mp4")
    caption_file = TRANSCRIPTION_ES / f"{stem}.json"

    with logfire.span("stitch", video=stem):
        stitch_audio(
            video_path  = str(video_files[0]),
            audio_path  = str(audio_files[0]),
            output_path = output_path,
        )
    print(f"Output: {output_path}")

In [ ]:
# Full path: burn subtitles (requires ImageMagick)
if video_files and audio_files and caption_file.exists():
    output_subs = str(OUTPUT_VIDEO_DIR / f"{stem}_es_subs.mp4")
    with logfire.span("stitch_with_subs", video=stem):
        stitch_video_with_timestamps(
            video_path   = str(video_files[0]),
            caption_path = str(caption_file),
            audio_path   = str(audio_files[0]),
            output_path  = output_subs,
        )
    print(f"With subtitles: {output_subs}")

### Play the final dubbed video

In [ ]:
from IPython.display import Video

output_files = list(OUTPUT_VIDEO_DIR.glob("*.mp4"))
if output_files:
    Video(str(output_files[0]), embed=True, width=640)

---
## Batch processing — all registered videos

Run the full pipeline for every video in `video_registry.yml`.

In [ ]:
import yaml
from download_video import download_video, download_caption
from transcribe import load_model, transcribe_videos
from translate_en_to_es import download_and_install_package, translate_all_files
from tts_es import text_file_to_speech, files_from_dir
from translated_output import stitch_audio

with open(REPO_ROOT / "video_registry.yml") as f:
    registry = yaml.safe_load(f)

with logfire.span("batch_pipeline", n_videos=len(registry["videos"])):

    for video in registry["videos"]:
        with logfire.span("download", video_id=video["id"]):
            download_video(video["url"], str(RAW_VIDEO_DIR))
            download_caption(video["url"], str(RAW_CAPTION_DIR))

    model = load_model("tiny")
    with logfire.span("transcribe_batch"):
        transcribe_videos(str(RAW_VIDEO_DIR), model, str(TRANSCRIPTION_EN))

    download_and_install_package("en", "es")
    with logfire.span("translate_batch"):
        translate_all_files(str(TRANSCRIPTION_EN), str(TRANSCRIPTION_ES))

    for es_file in files_from_dir(str(TRANSCRIPTION_ES)):
        with logfire.span("tts", source=pathlib.Path(es_file).name):
            text_file_to_speech(es_file, str(AUDIO_DIR))

    for video_file in RAW_VIDEO_DIR.glob("*.mp4"):
        audio_file = AUDIO_DIR / (video_file.stem + ".wav")
        if audio_file.exists():
            out = OUTPUT_VIDEO_DIR / (video_file.stem + "_es.mp4")
            with logfire.span("stitch", video=video_file.name):
                stitch_audio(str(video_file), str(audio_file), str(out))

print("Batch pipeline complete.")

---
## Summary

### Pipeline milestones

| Milestone | Tool | Output |
|-----------|------|--------|
| M1 Download | yt-dlp, youtube-transcript-api | `raw_videos/*.mp4`, `raw_captions/*.txt` |
| M1-align VAD | silero-vad | speech / silence timeline |
| M2-align Diarization | pyannote.audio | speaker-turn timeline |
| M2 Transcribe | openai-whisper | `transcriptions/en/*.json` |
| M3 Translate | argostranslate (OpenNMT, offline) | `transcriptions/es/*.json` |
| M3-align Metrics | `SegmentMetrics` dataclass | duration error, predicted stretch |
| M4-align Policy | `AlignAction` enum | accept / stretch / shift / retry / fail |
| M5-align Re-rank | PydanticAI Translation Agent _(opt)_ | shorter candidate translations |
| M6-align Global align | greedy timeline optimizer | scheduled segment starts/ends |
| M4 TTS | Coqui TTS / XTTS + pyrubberband | `audios/*.wav` |
| M5 Stitch | moviepy + ffmpeg | `output_videos/*.mp4` |
| M8-align Eval | clip metrics + Failure Agent _(opt)_ | `metrics/*.json` |

### PydanticAI agents (optional — `pip install pydantic-ai`)

| Agent | Milestone | Role |
|-------|-----------|------|
| Translation Candidate Agent | M5-align | Proposes shorter alternatives for over-budget segments |
| Alignment Policy Agent | M6-align | Selects among deterministic actions using measured costs |
| Failure Analysis Agent | M8-align | Clusters failure modes from evaluation reports |

### Logfire spans (optional — `pip install logfire`)

Every stage wraps its core call in `logfire.span(...)` with structured fields so any
segment decision can be inspected across runs.  The spans degrade gracefully to no-ops
when Logfire is not installed.

Recommended span fields per segment (from the architecture doc):
`segment_index`, `source_duration_ms`, `baseline_translation`, `predicted_stretch`,
`raw_tts_duration_ms`, `gap_shift_ms`, `cumulative_drift_ms`, `policy_action`, `speaker_id`

### Design principle

> Treat dubbing as constrained sequence generation under a temporal budget.
> Stop repairing timing at the end of `tts_es.py` and start treating it as
> information that flows through the whole pipeline.